# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells (FAIR^2) Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their IDs.


In [ ]:
# List record sets
print('Available record sets in the dataset:')
for rs in metadata.record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs.get('name', '(no name)')}")

# For each record set, list fields and columns by @id
for rs in metadata.record_sets:
    print(f"\nRecord Set @id: {rs['@id']}")
    print('  Fields:')
    for field in rs.get('field', []):
        if isinstance(field, dict):
            f_id = field.get('@id', str(field))
            name = field.get('name', '')
            print(f"    - @id: {f_id}, name: {name}")
        else:
            print(f"    - @id: {field}")
    print('  Columns:')
    for col in rs.get('column', []):
        if isinstance(col, dict):
            c_id = col.get('@id', str(col))
            name = col.get('name', '')
            print(f"    - @id: {c_id}, name: {name}")
        else:
            print(f"    - @id: {col}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.


In [ ]:
# Set up record set IDs (from section above, using @id)
# Fetch all recordSet @id's into a list
record_sets = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    # Load records for each record set by @id
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for Record Set: {record_set_id} - shape: {dataframes[record_set_id].shape}")
    else:
        print(f"No records found for Record Set: {record_set_id}")

# Example: examine the fields/columns from one record set
if len(record_sets) > 0:
    first_rs = record_sets[0]
    if first_rs in dataframes:
        print(f"Columns for Record Set {first_rs}:")
        print(dataframes[first_rs].columns.tolist())
        dataframes[first_rs].head()
    else:
        print(f"No dataframe loaded for first Record Set: {first_rs}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.


In [ ]:
# Choose the first available record set for demonstration (update if desired)
if len(dataframes):
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    print(f"Analyzing Record Set: {rs_id}")

    # Try to choose a numeric field for demo
    numeric_fields = df.select_dtypes('number').columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field '{numeric_field}' for filtering and normalization.")

        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to choose a categorical/group field
        object_fields = df.select_dtypes('object').columns.tolist()
        group_field = None
        for col in object_fields:
            # Choose group field with sufficient unique values
            if df[col].nunique() > 1:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped data by '{group_field}':")
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No DataFrame available for EDA.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes):
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    # Numeric field plotted
    numeric_fields = df.select_dtypes('number').columns.tolist()
    if numeric_fields:
        field_to_plot = numeric_fields[0]
        plt.figure(figsize=(6, 4))
        sns.histplot(df[field_to_plot].dropna(), kde=True, bins=30)
        plt.title(f"Distribution of {field_to_plot}")
        plt.xlabel(field_to_plot)
        plt.ylabel("Count")
        plt.show()

        # Correlation with another numeric field if available
        if len(numeric_fields) > 1:
            plt.figure(figsize=(6, 4))
            sns.scatterplot(data=df, x=numeric_fields[0], y=numeric_fields[1])
            plt.title(f"{numeric_fields[0]} vs {numeric_fields[1]}")
            plt.show()
    else:
        print("No numeric fields available for visualization.")
else:
    print("No DataFrame available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and perform basic analysis on the FAIR^2 dataset using the `mlcroissant` library. We reviewed available record sets and fields using their `@id`, extracted tabular data, performed filtering and normalization, grouped values, and visualized numeric distributions. This workflow establishes a foundation for more advanced bioinformatic and proteomics data exploration.
